In [0]:
%pip install category_encoders

In [0]:
dbutils.library.restartPython()

In [0]:
%pip install fastapi uvicorn scikit-learn joblib pydantic

In [0]:
# ================================
# IMPORTS
# ================================
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import pickle
import numpy as np
import uvicorn
from threading import Thread
import nest_asyncio

# Allow nested event loops (required for Databricks)
nest_asyncio.apply()

# ================================
# INITIALIZE FASTAPI APP
# ================================
app = FastAPI(title="Train Delay Prediction API")

# ================================
# LOAD DATASET FROM DBFS
# ================================
DATA_PATH = "workspace.default.route_optimisation_data"  # Correct path for dataset in DBFS
OPTIMIZED_ROUTE_TABLE = "workspace.default.route_optimisation_data"

try:
    df = pd.read_csv(DATA_PATH)
    
    # Ensure correct data types
    df["train_number"] = df["trainNumber"].astype(str)
    df["station_code"] = df["toStnCode"].astype(str)
    
    # Optimize lookup with multi-index (VERY IMPORTANT for performance)
    df.set_index(["train_number", "station_code"], inplace=True)
    print(f"✓ Loaded {len(df)} routes from dataset")
except Exception as e:
    print(f"✗ Error loading dataset: {e}")
    df = None

# ================================
# LOAD MODEL
# ================================
MODEL_PATH = "/Workspace/Repos/che240008012@iiti.ac.in/Databricks-Hackathon/Databricks 2.0/train_delay_predictor.pkl"

try:
    with open(MODEL_PATH, "rb") as f:
        model = pickle.load(f)
    print(f"✓ Model loaded successfully")
except Exception as e:
    print(f"✗ Error loading model: {e}")
    model = None

# ================================
# DEFINE REQUEST SCHEMA
# ================================
class PredictionRequest(BaseModel):
    station_code: str
    train_number: str
    month: int
    date: int

# ================================
# API ENDPOINTS
# ================================
@app.get("/")
def root():
    return {
        "message": "Train Delay Prediction API",
        "endpoints": {
            "/predict": "GET - Predict delay with query params",
            "/predict_delay": "POST - Predict delay with JSON body",
            "/health": "GET - Health check"
        }
    }

@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "model_loaded": model is not None,
        "data_loaded": df is not None,
        "routes_count": len(df) if df is not None else 0
    }

@app.get("/predict")
def predict_delay_get(
    station_code: str,
    train_number: str,
    month: int,
    date: int
):
    # Validate models are loaded
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    if df is None:
        raise HTTPException(status_code=503, detail="Dataset not loaded")
    
    # Validate input ranges
    if not (1 <= month <= 12):
        raise HTTPException(status_code=400, detail="Month must be between 1 and 12")
    if not (1 <= date <= 31):
        raise HTTPException(status_code=400, detail="Date must be between 1 and 31")
    if not station_code or not train_number:
        raise HTTPException(status_code=400, detail="Station code and train number are required")

    try:
        # Get distance using indexed lookup (O(1) complexity)
        distance = df.loc[(train_number, station_code), "distance"]
    except KeyError:
        raise HTTPException(
            status_code=404,
            detail=f"No route found for train {train_number} at station {station_code}"
        )

    # Prepare input for model
    features = np.array([[distance, month, date]])
    
    # Predict delay
    prediction = model.predict(features)[0]

    # Return response
    return {
        "train_number": train_number,
        "station_code": station_code,
        "distance": float(distance),
        "month": month,
        "date": date,
        "predicted_delay_minutes": float(prediction)
    }

@app.post("/predict_delay")
def predict_delay_post(request: PredictionRequest):
    # Reuse the GET endpoint logic
    return predict_delay_get(
        station_code=request.station_code,
        train_number=request.train_number,
        month=request.month,
        date=request.date
    )

# ================================
# RUN FASTAPI SERVER
# ================================
print("\n" + "="*50)
print("Starting FastAPI server...")
print("="*50)
print("API will be available at: http://0.0.0.0:8000")
print("Documentation at: http://0.0.0.0:8000/docs")
print("="*50 + "\n")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Run in background thread (required for Databricks notebooks)
server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

print("✓ Server started in background thread")

In [0]:
app = FastAPI()

In [0]:
%pip install fastapi uvicorn nest_asyncio

In [0]:
# =========================================
# IMPORTS
# =========================================
from fastapi import FastAPI, HTTPException
import pandas as pd
import pickle
import uvicorn
from threading import Thread

# =========================================
# INIT APP
# =========================================
app = FastAPI()

# =========================================
# LOAD TABLE 1: ROUTE SEARCH DATA
# =========================================
try:
    route_df = spark.read.table("workspace.default.route_optimisation_data").toPandas()
    print("✓ route_optimisation_data loaded")
except Exception as e:
    print(f"✗ Error loading route_optimisation_data: {e}")
    route_df = pd.DataFrame()

# =========================================
# LOAD TABLE 2: MODEL INPUT DATA
# =========================================
try:
    optimize_df = spark.read.table("workspace.default.optimize_route").toPandas()
    print("✓ optimize_routes loaded")
except Exception as e:
    print(f"✗ Error loading optimize_routes: {e}")
    optimize_df = pd.DataFrame()

# =========================================
# CLEAN TYPES
# =========================================
if not route_df.empty:
    route_df["train_number"] = route_df["trainNumber"].astype(str)

if not optimize_df.empty:
    optimize_df["train_number"] = optimize_df["trainNumber"].astype(str)

# =========================================
# LOAD MODEL (MODEL 1 ONLY)
# =========================================
try:
    with open("/Workspace/Repos/che240008012@iiti.ac.in/Databricks-Hackathon/Databricks 2.0/(Clone) train_delay_model.pkl", "rb") as f:
        delay_model = pickle.load(f)
    print("✓ Delay model loaded")
except Exception as e:
    print(f"✗ Error loading delay model: {e}")
    delay_model = None

# =========================================
# HELPER: GET DISTANCE + STATION NAME FROM optimize_routes
# =========================================
def get_model_features(train_number, station_name):

    filtered = optimize_df[
        (optimize_df["train_number"] == train_number) &
        (optimize_df["station_name"] == station_name)
    ]

    if filtered.empty:
        return None, None

    distance = filtered.iloc[0]["distance"]
    station = filtered.iloc[0]["toStnCode"]

    return distance, station

# =========================================
# HELPER: DELAY PREDICTION
# =========================================
def predict_delay(distance, month, date):

    if delay_model is None or distance is None:
        return 0.0

    input_data = [[distance, month, date]]
    prediction = delay_model.predict(input_data)

    return float(prediction[0])

# =========================================
# HELPER: SEARCH ROUTES (TABLE 1)
# =========================================
def search_routes(source, destination, min_price, max_price):

    filtered_df = route_df[
        (route_df["source"] == source) &
        (route_df["destination"] == destination)
    ]

    if filtered_df.empty:
        return []

    routes = []

    for _, row in filtered_df.iterrows():

        total_price = row["base_fare"] + row["railway_charges"] + row["tax"]

        if min_price <= total_price <= max_price:
            routes.append(row)

    return routes

# =========================================
# COMBINED API
# =========================================
@app.get("/smart-search")
def smart_search(
    source: str,
    destination: str,
    min_price: float,
    max_price: float,
    month: int,
    date: int
):

    # Validate input
    if min_price > max_price:
        raise HTTPException(status_code=400, detail="Invalid price range")

    if route_df.empty or optimize_df.empty:
        raise HTTPException(status_code=500, detail="Dataset not loaded")

    # Step 1: Get candidate routes
    routes_data = search_routes(source, destination, min_price, max_price)

    if not routes_data:
        raise HTTPException(status_code=404, detail="No routes found")

    routes = []

    # Step 2: Enrich with model 1
    for row in routes_data:

        train_number = row["train_number"]
        station_name = row["station_name"]

        # Get distance from optimize_routes table
        distance, station = get_model_features(train_number, station_name)

        # Predict delay
        delay = predict_delay(distance, month, date)

        total_price = row["base_fare"] + row["railway_charges"] + row["tax"]

        route = {
            "train_number": train_number,
            "station_name": station,
            "departure_time": row["departure_time"],
            "arrival_time": row["arrival_time"],
            "distance": float(distance) if distance else None,
            "total_price": float(total_price),
            "predicted_delay_minutes": delay
        }

        routes.append(route)

    # =====================================
    # RANKING
    # =====================================
    prices = [r["total_price"] for r in routes]
    delays = [r["predicted_delay_minutes"] for r in routes]

    min_price_val, max_price_val = min(prices), max(prices)
    min_delay, max_delay = min(delays), max(delays)

    price_range = max_price_val - min_price_val if max_price_val != min_price_val else 1
    delay_range = max_delay - min_delay if max_delay != min_delay else 1

    PRICE_WEIGHT = 0.6
    DELAY_WEIGHT = 0.4

    for r in routes:
        norm_price = (r["total_price"] - min_price_val) / price_range
        norm_delay = (r["predicted_delay_minutes"] - min_delay) / delay_range

        r["score"] = PRICE_WEIGHT * norm_price + DELAY_WEIGHT * norm_delay

    routes = sorted(routes, key=lambda x: x["score"])

    # Remove score
    for r in routes:
        r.pop("score")

    return {
        "total_routes_found": len(routes),
        "best_routes": routes[:5]
    }

# =========================================
# RUN SERVER
# =========================================
def run():
    uvicorn.run(app, host="0.0.0.0", port=8001)

thread = Thread(target=run)
thread.start()

In [0]:
spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW SCHEMAS IN workspace").show()
spark.sql("SHOW TABLES IN workspace.default").show()

In [0]:
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)